# Part 2: Understanding Your LLM

> **Goal:** Build precise intuition for how tokenization, context windows, and model selection affect the systems you can build.

In [169]:
# Imports

import time
import mlflow
import tiktoken
from chat.client import ChatClient
from utils.tracker import setup_mlflow
from utils.client import DATABRICKS_HOST, DATABRICKS_TOKEN
import pandas as pd
from IPython.display import display
import random
import uuid

## Task 2.1: Tokenization Deep Dive

### Implementation

In [170]:
prose = (
            "The rapid advancement of artificial intelligence has fundamentally altered "
            "the landscape of modern computing. Machine learning models, particularly "
            "large language models, possess billions of parameters and are trained on "
            "vast corpora of internet text. This extensive training enables them to "
            "perform a wide variety of tasks, from summarizing lengthy documents and "
            "translating between languages to writing functional computer code. However, "
            "these systems are not without their limitations. They can occasionally "
            "generate plausible-sounding but entirely incorrect information, a phenomenon "
            "commonly referred to as hallucination. Furthermore, the immense computational "
            "resources required to train and deploy these models raise important questions "
            "about their environmental impact and the accessibility of state-of-the-art "
            "technology. Researchers are continuously exploring new architectures, such as "
            "more efficient transformer variants, to mitigate these issues. As the field "
            "progresses, the focus is increasingly shifting towards building systems that "
            "are not only highly capable but also robust, interpretable, and aligned with "
            "human values. The next decade will likely witness even more profound changes "
            "as these technologies are integrated more deeply into our daily lives and "
            "critical infrastructure."
        )

code = (
            "def calculate_moving_average(data: list[float], window_size: int) -> list[float]:\n"
            "    \"\"\"\n"
            "    Computes the simple moving average of a list of numbers.\n"
            "    \n"
            "    Args:\n"
            "        data: List of numerical values.\n"
            "        window_size: The number of periods to average over.\n"
            "    \"\"\"\n"
            "    if len(data) < window_size:\n"
            "        return []\n"
            "    \n"
            "    # Calculate initial window sum\n"
            "    window_sum = sum(data[:window_size])\n"
            "    moving_averages = [window_sum / window_size]\n"
            "    \n"
            "    # Slide the window across the rest of the data\n"
            "    for i in range(len(data) - window_size):\n"
            "        window_sum = window_sum - data[i] + data[i + window_size]\n"
            "        moving_averages.append(window_sum / window_size)\n"
            "        \n"
            "    return moving_averages"
        )

json = (
            "{\n"
            "  \"company\": {\n"
            "    \"name\": \"TechCorp\",\n"
            "    \"departments\": [\n"
            "      {\n"
            "        \"name\": \"Engineering\",\n"
            "        \"teams\": {\n"
            "          \"backend\": {\n"
            "            \"lead\": \"Alice\",\n"
            "            \"members\": [\"Bob\", \"Charlie\"]\n"
            "          },\n"
            "          \"frontend\": {\n"
            "            \"lead\": \"David\",\n"
            "            \"members\": [\"Eve\", \"Frank\"]\n"
            "          }\n"
            "        }\n"
            "      }\n"
            "    ]\n"
            "  }\n"
            "}"
        )

md = (
            "# System Architecture\n\n"
            "This document outlines the core components of the system.\n\n"
            "## Requirements\n"
            "* High availability\n"
            "* Low latency\n"
            "* Fault tolerance\n\n"
            "## Deployment\n"
            "To deploy the application, run the following command:\n\n"
            "`docker-compose up --build -d`\n\n"
            "Ensure the `.env` file is properly configured before starting."
        )

math = (
            "The fundamental theorem of calculus connects differentiation and integration.\n"
            "Let $f$ be a continuous real-valued function defined on a closed interval $[a, b]$. "
            "Let $F$ be the function defined, for all $x$ in $[a, b]$, by:\n\n"
            "$$ F(x) = \\int_{a}^{x} f(t) \\, dt $$\n\n"
            "Then $F$ is uniformly continuous on $[a, b]$ and differentiable on the open interval $(a, b)$, "
            "and $F'(x) = f(x)$ for all $x$ in $(a, b)$. Furthermore, if $f$ is Riemann integrable, then:\n\n"
            "$$ \\int_{a}^{b} f(x) \\, dx = F(b) - F(a) $$"
        )

In [171]:
content_samples = {
        "Prose English": prose,
        "Python code": code,
        "JSON": json,
        "Markdown": md,
        "Mathematical notation": math 
    }


In [172]:
setup_mlflow("Task-2.1-Tokenization")

MLflow experiment set: /Users/28100445@lums.edu.pk/Task-2.1-Tokenization at databricks


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [173]:
encodings = {
    "cl100k_base": tiktoken.get_encoding("cl100k_base"),
    "o200k_base": tiktoken.get_encoding("o200k_base")
}

In [174]:
results = []

for content_name, content_text in content_samples.items():
    char_count = len(content_text)
    for enc_name, enc in encodings.items():
        token_count = len(enc.encode(content_text))
        ratio = char_count / token_count if token_count else 0
        results.append({
            "Content Type": content_name,
            "Tokenizer": enc_name,
            "Char Count": char_count,
            "Token Count": token_count,
            "Chars per Token": round(ratio, 2)
        })

with mlflow.start_run(run_name="tokenization-analysis"):
    for row in results:
        metric_name = f"{row['Content Type']}_{row['Tokenizer']}_chars_per_token".replace(" ", "_")
        mlflow.log_metric(metric_name, row["Chars per Token"])

🏃 View run tokenization-analysis at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587466/runs/554b1e6f93844b178ee94c445042aed6
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587466


In [175]:
# Display a summary table of the results.
results_df = pd.DataFrame(results)
display(results_df)

,Content Type,Tokenizer,Char Count,Token Count,Chars per Token
0,Prose English,cl100k_base,1302,204,6.38
1,Prose English,o200k_base,1302,204,6.38
2,Python code,cl100k_base,713,163,4.37
3,Python code,o200k_base,713,165,4.32
4,JSON,cl100k_base,361,91,3.97
5,JSON,o200k_base,361,92,3.92
6,Markdown,cl100k_base,314,63,4.98
7,Markdown,o200k_base,314,63,4.98
8,Mathematical notation,cl100k_base,495,166,2.98
9,Mathematical notation,o200k_base,495,165,3.00


### Analysis Questions

1. Which content type had the highest characters-per-token ratio (most efficient)? Explain why this makes sense given how subword tokenizers are trained on internet data.
2. Which content type had the lowest ratio (least efficient)? What practical implication does this have for how you'd design prompts that must include this content type?
3. The `cl100k_base` and `o200k_base` encodings use different vocabulary sizes. Did this produce meaningful differences in the ratios you measured? Which tokenizer was more efficient for which content types, and why?
4. Based on your measurements, estimate the maximum number of words of plain English prose that fit in a 4,096-token context window. Show your calculation.

## Task 2.2: Context Window: Needle in a haystack

### Implementation

In [176]:
enc = tiktoken.get_encoding('cl100k_base')

target_id = "ffffffff"
num_records = 2500
haystack_lines = []

while len(haystack_lines) < num_records:
    random_id = uuid.uuid4().hex[:8]
    if random_id != target_id:
        record = f"Record-ID: {random_id} | Clearance: {random.randint(1,5)} | Sector: {uuid.uuid4().hex[:6]}\n"
        haystack_lines.append(record)

print(f"No. of tokens in the haystack: {len(enc.encode(chr(10).join(haystack_lines)))}")

No. of tokens in the haystack: 54628


In [177]:

setup_mlflow("Task-2.2-Needle-in-a-haystack")

MLflow experiment set: /Users/28100445@lums.edu.pk/Task-2.2-Needle-in-a-haystack at databricks


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [178]:
trials = 3
recall_results = []

client = ChatClient(model="databricks-gemma-3-12b", budgeting=False) 

In [179]:
target_id = "ffffffff"
target_sector = "abc123"
needle = f"Record-ID: {target_id} | Clearance: 5 | Sector: {target_sector}\n"
question = f"What is the Sector value for Record-ID: {target_id}? Reply with only the Sector value, nothing else."

positions = ["beginning", "middle", "end"]

In [180]:
def insert_needle(haystack_lines, needle, position):
    lines = list(haystack_lines)
    n = len(lines)
    if position == "beginning":
        idx = int(n * 0.05)
    elif position == "middle":
        idx = int(n * 0.5)
    else:  # end
        idx = int(n * 0.95)
    lines.insert(idx, needle)
    return lines

In [181]:
# Insert needle into the haystack at positions

# Then run the query in MLflow

# Dont forget to log params and metrics
for position in positions:
    for trial in range(1, trials + 1):
        haystack_with_needle = insert_needle(haystack_lines, needle, position)
        haystack_text = "\n".join(haystack_with_needle)
        full_query = f"{haystack_text}\n\n{question}"

        client.reset_conversation()
        response = client.chat(full_query, stream=False)
        recalled = target_sector in response

        with mlflow.start_run(run_name=f"needle_{position}_trial{trial}"):
            mlflow.log_param("needle_position", position)
            mlflow.log_param("trial", trial)
            mlflow.log_param("haystack_tokens", len(enc.encode(haystack_text)))
            mlflow.log_metric("recall", int(recalled))

        recall_results.append({
            "Position": position,
            "Trial": trial,
            "Recalled": recalled,
            "Model Response": response
        })

# Display the results here as a table as well.
recall_df = pd.DataFrame(recall_results)
display(recall_df)

summary_df = recall_df.groupby("Position")["Recalled"].mean().reset_index()
summary_df.columns = ["Position", "Recall Rate"]
display(summary_df)

🏃 View run needle_beginning_trial1 at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587467/runs/1e431d6f4afe458eafa4c28db0da4d55
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587467
🏃 View run needle_beginning_trial2 at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587467/runs/f3eb39b8141b4bc891adde699351fdbe
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587467
🏃 View run needle_beginning_trial3 at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587467/runs/ed7dc68c64a84bb6ad20d36d34bfc0b0
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587467
🏃 View run needle_middle_trial1 at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587467/runs/7471beba7e964afb98b21d1ddcad8c55
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/expe

,Position,Trial,Recalled,Model Response
0,beginning,1,True,abc123
1,beginning,2,True,abc123
2,beginning,3,True,abc123
3,middle,1,True,abc123
4,middle,2,True,abc123
5,middle,3,True,abc123
6,end,1,True,abc123
7,end,2,True,abc123
8,end,3,True,abc123


,Position,Recall Rate
0,beginning,1.0
1,end,1.0
2,middle,1.0


### Analysis Questions
 
1. Report your recall results by position (e.g., beginning: 3/3, middle: 1/3, end: 3/3). Is there a positional effect? Is it consistent across your 3 trials per position?
2. Why might a model have structurally lower recall for facts placed at the middle of the context? What does this suggest about how attention is distributed across long sequences?
3. Describe a real system (e.g. a customer support bot, a code assistant, or a document Q&A tool) where such a recall failure would cause a **silent**, hard-to-diagnose bug. What would the failure look like to the end user?
4. Based on your findings, state one concrete design guideline you would apply when constructing context windows for a Robust Information Retrieval system.

## Task 2.3: Model Comparison

### Implementation

In [182]:
from databricks.sdk import WorkspaceClient

# Initializes client using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)
# List all endpoints
endpoints = w.serving_endpoints.list()
# Print endpoint details
for endpoint in endpoints:
    print(f"Name: {endpoint.name} | State: {endpoint.state.ready}")

# > Choose any 3 from Here


Name: databricks-claude-opus-4-8 | State: EndpointStateReady.READY
Name: databricks-gemini-3-5-flash | State: EndpointStateReady.READY
Name: databricks-gpt-oss-120b | State: EndpointStateReady.READY
Name: databricks-gpt-oss-20b | State: EndpointStateReady.READY
Name: databricks-qwen3-next-80b-a3b-instruct | State: EndpointStateReady.READY
Name: databricks-qwen35-122b-a10b | State: EndpointStateReady.READY
Name: databricks-llama-4-maverick | State: EndpointStateReady.READY
Name: databricks-gemma-3-12b | State: EndpointStateReady.READY
Name: databricks-gte-large-en | State: EndpointStateReady.READY
Name: databricks-bge-large-en | State: EndpointStateReady.READY
Name: databricks-meta-llama-3-1-8b-instruct | State: EndpointStateReady.READY
Name: databricks-meta-llama-3-3-70b-instruct | State: EndpointStateReady.READY
Name: databricks-qwen3-embedding-0-6b | State: EndpointStateReady.READY


In [183]:
models_to_test = [
    "databricks-meta-llama-3-3-70b-instruct",
    "databricks-meta-llama-3-1-8b-instruct",
    "databricks-gpt-oss-120b"    
]

# For the 3 models you chose, create a mapping of their costs. Look up the costs for a Pay-per-Token scheme in Databricks for them.
cost_rates = {
    "databricks-meta-llama-3-3-70b-instruct": 0.00100,
    "databricks-meta-llama-3-1-8b-instruct": 0.00030,
    "databricks-gpt-oss-120b": 0.00150
    }

In [184]:
test_query = "Explain the difference between TCP and UDP in 3 sentences."

In [185]:
setup_mlflow("Task-2.3-Model-Comparison")
benchmark_results = []

MLflow experiment set: /Users/28100445@lums.edu.pk/Task-2.3-Model-Comparison at databricks


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [186]:
for model_name in models_to_test:
    print(f"\nBenchmarking {model_name}...")

    bench_client = ChatClient(model=model_name, budgeting=False)
    bench_client.history.append({"role": "user", "content": test_query})

    start_time = time.time()
    first_token_time = None
    full_response = ""

    stream = bench_client.client.chat.completions.create(
        model=model_name,
        messages=bench_client.history,
        stream=True
    )

    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            if first_token_time is None:
                first_token_time = time.time()
                
            content = chunk.choices[0].delta.content
            
            # Case 1: Content comes back as a list of dictionaries or objects
            if isinstance(content, list):
                extracted_pieces = []
                for item in content:
                    if isinstance(item, dict):
                        # Try to grab common text keys, fallback to casting
                        text_piece = item.get("text") or item.get("content") or str(item)
                        extracted_pieces.append(text_piece)
                    else:
                        extracted_pieces.append(str(item))
                content_str = "".join(extracted_pieces)
                
            # Case 2: Content comes back directly as a single dictionary
            elif isinstance(content, dict):
                content_str = content.get("text") or content.get("content") or ""
                
            # Case 3: Content is a clean, native string
            else:
                content_str = str(content)
                
            full_response += content_str

    end_time = time.time()

    ttft = (first_token_time - start_time) if first_token_time else None
    total_latency = end_time - start_time
    output_tokens = bench_client.get_token_count(full_response)
    cost = output_tokens * cost_rates.get(model_name, 0) / 1000

    sentences = [s.strip() for s in full_response.replace('!', '.').replace('?', '.').split('.') if s.strip()]
    has_tcp = "tcp" in full_response.lower()
    has_udp = "udp" in full_response.lower()

    if len(sentences) == 3 and has_tcp and has_udp:
        quality_score = 5
    elif abs(len(sentences) - 3) <= 1 and (has_tcp or has_udp):
        quality_score = 4
    elif has_tcp or has_udp:
        quality_score = 3
    elif len(sentences) > 0:
        quality_score = 2
    else:
        quality_score = 1

    with mlflow.start_run(run_name=f"benchmark-{model_name}"):
        mlflow.log_param("model", model_name)
        mlflow.log_metric("ttft_sec", ttft if ttft is not None else -1)
        mlflow.log_metric("total_latency_sec", total_latency)
        mlflow.log_metric("output_tokens", output_tokens)
        mlflow.log_metric("quality_score", quality_score)
        mlflow.log_metric("cost_proxy_usd", cost)

    benchmark_results.append({
        "Model": model_name,
        "TTFT (s)": round(ttft, 3) if ttft is not None else None,
        "Total Latency (s)": round(total_latency, 3),
        "Output Tokens": output_tokens,
        "Quality (1-5)": quality_score,
        "Cost Proxy ($)": round(cost, 6)
    })

# Display a summary table of the results here as well.
benchmark_df = pd.DataFrame(benchmark_results)
display(benchmark_df)


Benchmarking databricks-meta-llama-3-3-70b-instruct...
🏃 View run benchmark-databricks-meta-llama-3-3-70b-instruct at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587468/runs/a9080990e8f54ac3b58e47f5c32cec6b
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587468

Benchmarking databricks-meta-llama-3-1-8b-instruct...
🏃 View run benchmark-databricks-meta-llama-3-1-8b-instruct at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587468/runs/35123fdabf7646c59f2f2182bb05cd7f
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587468

Benchmarking databricks-gpt-oss-120b...
🏃 View run benchmark-databricks-gpt-oss-120b at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587468/runs/5e42778d76f846f6b3f86ea009d36896
🧪 View experiment at: https://dbc-4641abd0-a3b8.cloud.databricks.com/ml/experiments/3881027723587468


,Model,TTFT (s),Total Latency (s),Output Tokens,Quality (1-5),Cost Proxy ($)
0,databricks-meta-llama-3-3-70b-instruct,1.638,3.322,115,5,0.000115
1,databricks-meta-llama-3-1-8b-instruct,1.529,2.053,133,5,0.000040
2,databricks-gpt-oss-120b,1.610,1.894,277,3,0.000416


### Analysis Questions
 
1. If TTFT is 3× worse for the larger model but quality is only marginally better, describe a specific use case where you would still pick the larger model. Then describe a use case where you would always pick the smaller model regardless of quality.
2. What does TTFT measure that total latency does not? Name a user-facing product feature where TTFT is the more important metric.
3. Your cost proxy is an estimate. Name two real-world pricing factors it ignores that would significantly affect the true cost in production.